# Plot Trial-Averaged Decoder Errors (Train on Cut, Predict on Uncut)

This notebook trains Ridge decoders exclusively on the steady-state locomotion portion of treadmill trials (excluding the initial acceleration period) and predicts targets on the whole trials (uncut). The resulting predictions and error traces are trial-averaged, aggregated across sessions for Layer 2/3 and Layer 5, and plotted separately for Running Speed, Optic Flow, and Depth.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import matplotlib.font_manager as fm

import flexiznam as flz
from cottage_analysis.summary_analysis import get_session_list
from cottage_analysis.analysis import treadmill

In [ ]:
# Set style
sns.set_theme(style="ticks", context="talk")
font_path = '/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'

SESSIONS_TO_EXCLUDE = {
    "PZAG22.1b_S20260220": "1000 more frames than triggers in the treadmill recording"
}

In [ ]:
def safe_log(arr):
    target = np.array(arr, dtype=float)
    invalid = target <= 0
    target[invalid] = np.nan
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.log(target)

def load_session_trials(sess, project, flexilims_session):
    filter_datasets = {"anatomical_only": 3, "ast_neuropil": False}
    annotated_filter = dict(filter_datasets)
    annotated_filter["annotated"] = True
    
    suite2p_datasets = flz.get_datasets(
        origin_name=sess,
        dataset_type="suite2p_rois",
        project_id=project,
        flexilims_session=flexilims_session,
        return_dataseries=False,
        filter_datasets={"anatomical_only": 3},
    )
    if not suite2p_datasets:
        raise FileNotFoundError(f"No suite2p_rois dataset found for {sess}")
    frame_rate = suite2p_datasets[0].extra_attributes["fs"]
    
    try:
        vs_df, trials_df = treadmill.sync_all_recordings(
            session_name=sess,
            flexilims_session=flexilims_session,
            project=project,
            filter_datasets=annotated_filter,
            recording_type="two_photon",
            photodiode_protocol=5,
            return_volumes=True,
            acceleration_time=None,
        )
    except FileNotFoundError:
        vs_df, trials_df = treadmill.sync_all_recordings(
            session_name=sess,
            flexilims_session=flexilims_session,
            project=project,
            filter_datasets=filter_datasets,
            recording_type="two_photon",
            photodiode_protocol=5,
            return_volumes=True,
            acceleration_time=None,
        )
        
    return trials_df, frame_rate

In [ ]:
def train_cut_predict_nocut(trials_df_nocut, target_col, frame_rate=15.0, k_folds=5, random_state=42):
    from sklearn.model_selection import KFold
    from sklearn.linear_model import Ridge
    from sklearn.preprocessing import StandardScaler

    acc_frames_list = []
    for _, trial in trials_df_nocut.iterrows():
        motor_speed = trial["expected_RS"] / 100.0
        acc_time = 0.13
        acc_frames = int((acc_time * motor_speed + 0.5) * frame_rate)
        acc_frames_list.append(acc_frames)
    trials_df_nocut["acc_frames"] = acc_frames_list

    first_val = trials_df_nocut[target_col].iloc[0]
    if not isinstance(first_val, (np.ndarray, list, pd.Series)):
        trials_df_nocut[target_col] = trials_df_nocut.apply(
            lambda r: np.repeat(r[target_col], len(r["dff_stim"])), axis=1
        )

    kf = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)
    trial_indices = np.arange(len(trials_df_nocut))
    y_pred_nocut = pd.Series(index=trials_df_nocut.index, dtype=object)

    for train_idx, test_idx in kf.split(trial_indices):
        X_train_list = []
        y_train_list = []
        for idx in train_idx:
            row = trials_df_nocut.iloc[idx]
            cut_idx = row["acc_frames"]
            X_train_list.append(row["dff_stim"][cut_idx:])
            y_train_list.append(row[target_col][cut_idx:])

        X_train = np.vstack(X_train_list)
        y_train = np.hstack(y_train_list)

        valid_train = ~np.isnan(y_train) & ~np.any(np.isnan(X_train), axis=1)
        X_train = X_train[valid_train]
        y_train = y_train[valid_train]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)

        model = Ridge(alpha=100.0)
        model.fit(X_train, y_train)

        for idx in test_idx:
            row = trials_df_nocut.iloc[idx]
            X_test = row["dff_stim"]
            valid_test = ~np.any(np.isnan(X_test), axis=1)
            X_test_valid = X_test[valid_test]
            y_pred_trial = np.full(len(X_test), np.nan)
            if len(X_test_valid) > 0:
                X_test_scaled = scaler.transform(X_test_valid)
                y_pred_trial[valid_test] = model.predict(X_test_scaled)
            y_pred_nocut.at[trials_df_nocut.index[idx]] = y_pred_trial

    return y_pred_nocut

In [ ]:
# Load all sessions across projects and run custom decoders
projects = ["ccyp_l5_3d_vision", "colasa_3d-vision_revisions"]
all_trial_averaged = []

for project in projects:
    flexilims_session = flz.get_flexilims_session(project_id=project)
    sessions = get_session_list.get_motor_session_list(flexilims_session)
    project_sessions = flz.get_entities("session", flexilims_session=flexilims_session)
    
    for sess in sessions:
        if sess in SESSIONS_TO_EXCLUDE:
            continue
        
        # Get nominal depth
        if sess in project_sessions.index:
            nominal_depth = project_sessions.loc[sess, "nominal_depth"]
            if isinstance(nominal_depth, (list, np.ndarray, pd.Series)):
                nominal_depth = np.mean(nominal_depth)
        else:
            continue
            
        try:
            print(f"Processing {sess}...")
            trials_df_nocut, frame_rate = load_session_trials(sess, project, flexilims_session)
            
            # Keep only closed loop trials
            trials_df_nocut = trials_df_nocut[trials_df_nocut.closed_loop == 1].copy()
            if len(trials_df_nocut) == 0:
                continue
                
            trials_df_nocut["expected_OF"] = trials_df_nocut.expected_optic_flow_stim.map(np.nanmedian)
            trials_df_nocut["expected_RS"] = trials_df_nocut.MotorSpeed_stim.map(np.nanmedian)
            
            # Fit decoders and make predictions
            # Target targets:
            # 1. RS_stim (using log target)
            trials_df_nocut["RS_stim_log"] = trials_df_nocut["RS_stim"].map(safe_log)
            rs_pred = train_cut_predict_nocut(trials_df_nocut, "RS_stim_log", frame_rate=frame_rate)
            trials_df_nocut["RS_pred"] = rs_pred
            
            # Calculate squared error
            rs_errs = []
            for rs_log, pred in zip(trials_df_nocut["RS_stim_log"], trials_df_nocut["RS_pred"]):
                if pred is None or (isinstance(pred, float) and np.isnan(pred)):
                    rs_errs.append(np.nan)
                else:
                    rs_errs.append((rs_log - pred) ** 2)
            trials_df_nocut["RS_error"] = rs_errs
            
            # 2. OF_stim (using log target)
            trials_df_nocut["OF_stim_log"] = trials_df_nocut["OF_stim"].map(safe_log)
            of_pred = train_cut_predict_nocut(trials_df_nocut, "OF_stim_log", frame_rate=frame_rate)
            trials_df_nocut["OF_pred"] = of_pred
            
            of_errs = []
            for of_log, pred in zip(trials_df_nocut["OF_stim_log"], trials_df_nocut["OF_pred"]):
                if pred is None or (isinstance(pred, float) and np.isnan(pred)):
                    of_errs.append(np.nan)
                else:
                    of_errs.append((of_log - pred) ** 2)
            trials_df_nocut["OF_error"] = of_errs
            
            # 3. depth (using log target)
            trials_df_nocut["depth_log"] = trials_df_nocut["depth"].map(safe_log)
            depth_pred = train_cut_predict_nocut(trials_df_nocut, "depth_log", frame_rate=frame_rate)
            trials_df_nocut["depth_pred"] = depth_pred
            
            depth_errs = []
            for d_log, pred in zip(trials_df_nocut["depth_log"], trials_df_nocut["depth_pred"]):
                if pred is None or (isinstance(pred, float) and np.isnan(pred)):
                    depth_errs.append(np.nan)
                else:
                    depth_errs.append((d_log - pred) ** 2)
            trials_df_nocut["depth_error"] = depth_errs
            
            # Average traces within session
            for (of_val, rs_val), tdf in trials_df_nocut.groupby(["expected_OF", "expected_RS"]):
                if len(tdf) == 0:
                    continue
                
                averaged_row = {
                    "session": sess,
                    "nominal_depth": nominal_depth,
                    "expected_OF": of_val,
                    "expected_RS": rs_val,
                }
                
                for col in ["RS_stim", "RS_error", "OF_error", "depth_error"]:
                    valid_arrays = [
                        arr for arr in tdf[col].values
                        if isinstance(arr, (np.ndarray, list, pd.Series))
                    ]
                    if len(valid_arrays) == 0:
                        averaged_row[f"{col}_mean"] = np.nan
                        averaged_row[f"{col}_sem"] = np.nan
                        continue
                        
                    min_len = min(len(arr) for arr in valid_arrays)
                    stacked = np.stack([arr[:min_len] for arr in valid_arrays])
                    averaged_row[f"{col}_mean"] = np.nanmean(stacked, axis=0)
                    
                all_trial_averaged.append(averaged_row)
        except Exception as e:
            print(f"Error processing session {sess}: {e}")

df_all = pd.DataFrame(all_trial_averaged)
cut_off = 350
df_all["layer"] = np.where(df_all["nominal_depth"] <= cut_off, "Layer 2/3", "Layer 5")

In [ ]:
# Aggregate across sessions for plotting
averaged_rows = []
for (layer, of, speed), group_df in df_all.groupby(["layer", "expected_OF", "expected_RS"]):
    if len(group_df) == 0:
        continue
    
    trace_cols = ["RS_stim", "RS_error", "OF_error", "depth_error"]
    row_data = {
        "layer": layer,
        "expected_OF": of,
        "expected_RS": speed,
        "n_sessions": len(group_df)
    }
    
    for col in trace_cols:
        valid_arrays = [
            arr for arr in group_df[f"{col}_mean"].values
            if isinstance(arr, (np.ndarray, list, pd.Series))
        ]
        if len(valid_arrays) == 0:
            row_data[f"{col}_mean"] = np.nan
            row_data[f"{col}_sem"] = np.nan
            continue
            
        min_len = min(len(arr) for arr in valid_arrays)
        stacked = np.stack([arr[:min_len] for arr in valid_arrays])
        row_data[f"{col}_mean"] = np.nanmean(stacked, axis=0)
        row_data[f"{col}_sem"] = stats.sem(stacked, axis=0, nan_policy='omit')
        
    averaged_rows.append(row_data)

averaged_df = pd.DataFrame(averaged_rows)

In [ ]:
# Plotting identical 3 figures with descending OF rows
unique_speeds = np.sort(averaged_df['expected_RS'].dropna().unique())
unique_ofs = np.sort(averaged_df['expected_OF'].dropna().unique())[::-1]

n_rows = len(unique_ofs)
n_cols = len(unique_speeds)

error_metrics = [
    ("RS_error", "Running Speed Decoder Error (Train on Cut, Predict on Uncut)"),
    ("OF_error", "Optic Flow Decoder Error (Train on Cut, Predict on Uncut)"),
    ("depth_error", "Depth Decoder Error (Train on Cut, Predict on Uncut)")
]

layer_colors = {
    "Layer 2/3": "#1a5276",
    "Layer 5": "#b03a2e"
}
layer_styles = {
    "Layer 2/3": "-",
    "Layer 5": "-"
}

for col_name, title in error_metrics:
    fig, axes = plt.subplots(
        nrows=n_rows, 
        ncols=n_cols, 
        figsize=(3.4 * n_cols, 2.7 * n_rows), 
        sharex=True, 
        sharey=True
    )
    
    if n_rows == 1 and n_cols == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = np.expand_dims(axes, axis=0)
    elif n_cols == 1:
        axes = np.expand_dims(axes, axis=1)
        
    shared_ax_rs = None
    
    for row_idx, of in enumerate(unique_ofs):
        for col_idx, speed in enumerate(unique_speeds):
            ax = axes[row_idx, col_idx]
            subset = averaged_df[
                (averaged_df['expected_RS'] == speed) & 
                (averaged_df['expected_OF'] == of)
            ]
            
            ax_rs = ax.twinx()
            if shared_ax_rs is None:
                shared_ax_rs = ax_rs
            else:
                ax_rs.sharey(shared_ax_rs)
                
            for layer in ["Layer 2/3", "Layer 5"]:
                layer_subset = subset[subset["layer"] == layer]
                if len(layer_subset) == 0:
                    continue
                row = layer_subset.iloc[0]
                
                rs_mean = row['RS_stim_mean']
                if isinstance(rs_mean, np.ndarray):
                    rs_style = ":" if layer == "Layer 5" else "--"
                    ax_rs.plot(rs_mean, color='black', linestyle=rs_style, alpha=0.15, label=f'Mean RS ({layer})')
                
                mean_trace = row[f"{col_name}_mean"]
                sem_trace = row[f"{col_name}_sem"]
                
                if isinstance(mean_trace, np.ndarray):
                    color = layer_colors[layer]
                    style = layer_styles[layer]
                    timepoints = np.arange(len(mean_trace))
                    ax.plot(timepoints, mean_trace, color=color, linestyle=style, linewidth=2.5, label=layer)
                    ax.fill_between(
                        timepoints,
                        mean_trace - sem_trace,
                        mean_trace + sem_trace,
                        color=color,
                        alpha=0.15 if layer == "Layer 2/3" else 0.08
                    )
            
            if col_idx < n_cols - 1:
                ax_rs.yaxis.set_ticklabels([])
            else:
                if row_idx == 0:
                    ax_rs.set_ylabel('Speed (m/s)', fontsize=10)
                    
            if row_idx == 0:
                ax.set_title(f"expected RS: {speed} cm/s", fontsize=11, fontweight='bold', pad=8)
                
            if col_idx == n_cols - 1:
                ax.text(
                    1.28, 0.5, f"expected OF:\n{of}", 
                    transform=ax.transAxes, 
                    va='center', ha='left', fontsize=10,
                    fontweight='semibold'
                )
                
            ax.axhline(0, color='grey', zorder=-2, linestyle=':')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            
    handles, labels = axes[0, 0].get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ordered_labels = ["Layer 2/3", "Layer 5"]
    ordered_handles = [by_label[lbl] for lbl in ordered_labels if lbl in by_label]
    axes[0, 0].legend(ordered_handles, ordered_labels, loc='upper right', frameon=False, fontsize=10)

    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    fig.text(0.5, 0.01, 'Frames', ha='center', fontsize=12)
    fig.text(0.01, 0.5, 'Decoder Error ($R^2$ or MSE)', va='center', rotation='vertical', fontsize=12)

    plt.tight_layout()
    plt.show()